# 5.0 — Data Preparation (v2)

## Changes from v1 (based on professor feedback)
- Reduced to 3 core clinical variables: UPDRS-III (OFF), MoCA, Hoehn & Yahr
- UPDRS-III filtered to OFF state only (PDSTATE == 3.0 or NaN)
- SC(Screening)↔BL(Baseline) EVENT_ID cross-mapping recovers ~119 additional patients
- Manual summation retained (NP3TOT not present in this PPMI download)
- UPDRS-IV, RBD, disease duration dropped

## 1. Imports and Configuration

In [2]:

import pandas as pd
import numpy as np
import pickle
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

In [11]:
# File paths 
LATENT_FILE    = '../../data/baseline/final_train_combined_vae_data.csv'
CLINICAL_DIR   = '../../data/raw/ppmi_clinical/'
UPDRS_III_FILE = os.path.join(CLINICAL_DIR, 'MDS-UPDRS_Part_III_06Jul2026.csv')
MOCA_FILE      = os.path.join(CLINICAL_DIR, 'Montreal_Cognitive_Assessment__MoCA_-Archived_25Jun2026.csv')

TRAIN_OUT      = '../../data/processed/clinical_merged/train_v2.csv'
VAL_OUT        = '../../data/processed/clinical_merged/val_v2.csv'
SCALER_OUT     = '../../results/models/scaler_sbr_v2.pkl'
PCA_OUT        = '../../results/models/pca_sbr_v2.pkl'

os.makedirs('../../data/processed/clinical_merged', exist_ok=True)
os.makedirs('../../results/models', exist_ok=True)

# ── Parameters ─────────────────────────────────────────────────────────────────
TRAIN_RATIO    = 0.8
RANDOM_STATE   = 42
JOIN_KEY       = ['PATNO', 'EVENT_ID']
PATIENT_COL    = 'PATNO'
LABEL_COL      = 'label'
SBR_COLS = [
    'DATSCAN_CAUDATE_R', 'DATSCAN_CAUDATE_L',
    'DATSCAN_PUTAMEN_R', 'DATSCAN_PUTAMEN_L',
    'DATSCAN_PUTAMEN_R_ANT', 'DATSCAN_PUTAMEN_L_ANT'
]
N_SBR_PCS = 3

print("Configuration loaded.")

Configuration loaded.


## 2. Load Latent Vectors

In [4]:
df_latents = pd.read_csv(LATENT_FILE)

print("Latent vectors")
print(f"Shape:           {df_latents.shape}")
print(f"Unique patients: {df_latents[PATIENT_COL].nunique()}")
print(f"Total rows:      {len(df_latents)}")
print(f"\nLabel counts:")
print(df_latents[LABEL_COL].value_counts().to_string())
print(f"\nEVENT_ID distribution (top 10):")
print(df_latents['EVENT_ID'].value_counts().head(10).to_string())

Latent vectors
Shape:           (2373, 303)
Unique patients: 1437
Total rows:      2373

Label counts:
label
PD         2030
Control     233
SWEDD       110

EVENT_ID distribution (top 10):
EVENT_ID
SC     1228
V06     392
V04     385
V10     256
U01      41
ST       32
V02      23
V05      10
U02       6


## 3. Load and Preprocess UPDRS-III
#### OFF state only

In [10]:
df_updrs3_raw = pd.read_csv(UPDRS_III_FILE)

print("UPDRS-III raw")
print(f"Shape: {df_updrs3_raw.shape}")
print(f"\nPDSTATE distribution:")
print(df_updrs3_raw['PDSTATE'].value_counts(dropna=False).to_string())
print(f"\nEVENT_ID distribution (top 10):")
print(df_updrs3_raw['EVENT_ID'].value_counts().head(10).to_string())

UPDRS-III raw
Shape: (38626, 65)

PDSTATE distribution:
PDSTATE
NaN    21372
ON      9802
OFF     7452

EVENT_ID distribution (top 10):
EVENT_ID
BL     5382
V04    4356
V06    3771
V08    2598
V05    2239
V02    2147
V10    1832
V12    1416
SC     1171
V14    1099


## 4. Load MoCA

In [12]:
df_moca_raw = pd.read_csv(MOCA_FILE)

# MCATOT is the precomputed total score (0-30)
df_moca = df_moca_raw[JOIN_KEY + ['MCATOT']].dropna(subset=['MCATOT'])
df_moca = df_moca.rename(columns={'MCATOT': 'MOCA_TOTAL'})

print("MoCA")
print(f"Rows:            {len(df_moca)}")
print(f"Unique patients: {df_moca[PATIENT_COL].nunique()}")
print(f"Score range:     {df_moca['MOCA_TOTAL'].min():.0f} – {df_moca['MOCA_TOTAL'].max():.0f}")
print(f"Mean ± std:      {df_moca['MOCA_TOTAL'].mean():.1f} ± {df_moca['MOCA_TOTAL'].std():.1f}")
print(f"\nEVENT_ID distribution (top 5):")
print(df_moca_raw['EVENT_ID'].value_counts().head(5).to_string())

MoCA
Rows:            7854
Unique patients: 2177
Score range:     0 – 30
Mean ± std:      26.6 ± 3.1

EVENT_ID distribution (top 5):
EVENT_ID
SC     1728
V04    1287
V06    1250
V08     881
V10     736


## 5. Merge Clinical Data with Latent Vectors
**SC(Screening)↔BL(Baseline) cross-mapping fix:**  
At enrollment, DaTSCAN is often acquired at EVENT_ID = SC while UPDRS/MoCA  
are measured at EVENT_ID = BL (or vice versa). A two-step merge recovers  
these patients: first exact match, then SC↔BL remapped match for unmatched rows.
